This file is used to generate the 100000 pos_neg_pairs required for training as per dictated by SC3000_CZ3005 Lab Assignment.pdf

Assumptions: 
Arguments in the questions must be 1 to 2 digit nonnegative integers
Allow 3 digit and negative answers

Things to note:
Generation max_new_tokens = 200
training max_length = 64 -> make sure positive/negative examples can't exceed 64

Worst case string:
x/99=99,x=? The answer is 9801 because 99*99 equals to 9801.
has 60 characters, +\n\n\n\n characters added in training = 64 chars, the training max length 

In [23]:
import inspect
import json
import math
import random
from abc import ABC, abstractmethod
from collections.abc import Callable
from functools import partial

minimum_samples = 100000

seed = 42
random.seed(seed)

# Binop Machinery

In [24]:
def generate_random_int_ne(target: int) -> int :
    p_choose_nearby_number = 0.9 # Either choose nearby number or uniform distribution from 0 to 99
    if random.random() < p_choose_nearby_number:
        candidate_choices = list(range(-40, 41))
        candidate_choices.remove(0)
        selected_choice = random.choice(candidate_choices)
        potential_ans = target + selected_choice
        if len(str(potential_ans)) > 4:
            return target - 1
        else:
            return potential_ans
    else:
        candidate_choices = list(range(100))
        if target in candidate_choices:
            candidate_choices.remove(target)
        return random.choice(candidate_choices)

class Binop(ABC):
    registry : dict = {}

    def __init__(self, op1: int, op2:int):
        self.result = self.execute(op1, op2)

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Binop.registry[cls.__name__] = cls

    def __str__(self):
        return str(self.result)

    def __repr__(self):
        return str(self.result)
    
    @classmethod
    @abstractmethod
    def symbol(cls) -> str:
        raise NotImplementedError("Subclass must implement abstract method")

    @classmethod
    @abstractmethod
    def execute(cls, op1: int, op2: int) -> int:
        raise NotImplementedError("Subclass must implement abstract method")

    @classmethod
    @abstractmethod
    def solve_for_op1(cls, op2: int, result: int) -> int:
        raise NotImplementedError("Subclass must implement abstract method")

    @classmethod
    @abstractmethod
    def solve_for_op2(cls, op1: int, result: int) -> int:
        raise NotImplementedError("Subclass must implement abstract method")

    @classmethod
    @abstractmethod
    def solve_for_op1_expanded(cls, op2: int, result: int) -> str:
        """
        Example:
            >>> Add.solve_for_op1_expanded(1, 2) # x + 1 = 2
            "2-1"
        """
        raise NotImplementedError("Subclass must implement abstract method")

    @classmethod
    @abstractmethod
    def solve_for_op2_expanded(cls, op1: int, result: int) -> str:
        """
        Example:
            >>> Add.solve_for_op2_expanded(1, 2) # 1 + x = 2
            "2-1"
        """
        raise NotImplementedError("Subclass must implement abstract method")

    @classmethod
    def generate_no_rearrange_problem(cls, op1: int, op2:int) -> str :
        return f"{op1}{cls.symbol()}{op2}=?"

    @classmethod
    def generate_no_rearrange_solution(cls, op1: int, op2:int) -> str :
        expanded_expression = f"{op1}{cls.symbol()}{op2}"
        collapsed_expression = f"{cls.execute(op1, op2)}"
        return f"{cls.generate_no_rearrange_problem(op1, op2)} The answer is {collapsed_expression} because {expanded_expression} equals {collapsed_expression}."


    @classmethod
    def generate_no_rearrange_solution_incorrect(cls, op1: int, op2:int) -> str :
        correct_ans = cls.execute(op1, op2)
        false_ans = generate_random_int_ne(correct_ans)
        expanded_expression = f"{op1}{cls.symbol()}{op2}"
        return f"{cls.generate_no_rearrange_problem(op1, op2)} The answer is {false_ans} because {expanded_expression} equals {false_ans}."

    @classmethod
    def generate_no_rearrange_solution_incorrect_str(cls, op1: int, op2:int) -> str :
        return f"{cls.generate_no_rearrange_problem(op1, op2)} Sorry, I do not know"
    
    @classmethod 
    def generate_x_problem(cls, x_first : bool, op: int, result: int) -> str:
        if x_first:
            return f"x{cls.symbol()}{op}={result},x=?"
        else:
            return f"{op}{cls.symbol()}x={result},x=?"

    @classmethod
    def generate_x_solution(cls, x_first : bool, op: int, result: int) -> str:
        if x_first:
            x = cls.solve_for_op1(op2=op, result=result)
            #1*x=6,x=? The answer is 6 because 6/1 equals to 6.
            return f"{cls.generate_x_problem(x_first, op, result)} The answer is {x} because {cls.solve_for_op1_expanded(op, result)} equals to {x}."
        else:
            x = cls.solve_for_op2(op1=op, result=result)
            return f"{cls.generate_x_problem(x_first, op, result)} The answer is {x} because {cls.solve_for_op2_expanded(op, result)} equals to {x}."

    @classmethod
    def generate_x_solution_incorrect(cls, x_first : bool, op: int, result: int) -> str:
        if x_first:
            x = cls.solve_for_op1(op2=op, result=result)
            false_x = generate_random_int_ne(x)
            return f"{cls.generate_x_problem(x_first, op, result)} The answer is {false_x} because {cls.solve_for_op1_expanded(op, result)} equals to {false_x}."
        else:
            x = cls.solve_for_op2(op1=op, result=result)
            false_x = generate_random_int_ne(x)
            return f"{cls.generate_x_problem(x_first, op, result)} The answer is {false_x} because {cls.solve_for_op2_expanded(op, result)} equals to {false_x}."

    @classmethod
    def generate_x_solution_incorrect_str(cls, x_first : bool, op: int, result: int) -> str:
        return f"{cls.generate_x_problem(x_first, op, result)} Sorry, I do not know"

In [25]:
class Add(Binop): 
    @classmethod
    def symbol(cls) -> str:
        return "+"

    @classmethod
    def execute(cls, op1: int, op2: int) -> int:
        return op1 + op2

    @classmethod
    def solve_for_op1(cls, op2: int, result: int) -> int:
        return result - op2

    @classmethod
    def solve_for_op2(cls, op1: int, result: int) -> int:
        return result - op1

    @classmethod
    def solve_for_op1_expanded(cls, op2: int, result: int) -> str:
        return f"{result}-{op2}"

    @classmethod
    def solve_for_op2_expanded(cls, op1: int, result: int) -> str:
        return f"{result}-{op1}"

class Minus(Binop): 
    @classmethod
    def symbol(cls) -> str:
        return "-"

    @classmethod
    def execute(cls, op1: int, op2: int) -> int:
        return op1 - op2

    @classmethod
    def solve_for_op1(cls, op2: int, result: int) -> int:
        return result + op2

    @classmethod
    def solve_for_op2(cls, op1: int, result: int) -> int:
        return op1 - result 

    @classmethod
    def solve_for_op1_expanded(cls, op2: int, result: int) -> str:
        return f"{result}+{op2}"

    @classmethod
    def solve_for_op2_expanded(cls, op1: int, result: int) -> str:
        return f"{op1}-{result}"

class Multiply(Binop): 
    @classmethod
    def symbol(cls) -> str:
        return "*"

    @classmethod
    def execute(cls, op1: int, op2: int) -> int:
        return op1 * op2

    @classmethod
    def solve_for_op1(cls, op2: int, result: int) -> int:
        if result % op2 != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        if op2 == 0:
            raise ArithmeticError("Division by zero unsupported")
        return result // op2

    @classmethod
    def solve_for_op2(cls, op1: int, result: int) -> int:
        if result % op1 != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        if op1 == 0:
            raise ArithmeticError("Division by zero unsupported")
        return result // op1

    @classmethod
    def solve_for_op1_expanded(cls, op2: int, result: int) -> str:
        if result % op2 != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        if op2 == 0:
            raise ArithmeticError("Division by zero unsupported")
        return f"{result}/{op2}"

    @classmethod
    def solve_for_op2_expanded(cls, op1: int, result: int) -> str:
        if result % op1 != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        if op1 == 0:
            raise ArithmeticError("Division by zero unsupported")
        return f"{result}/{op1}"

class Divide(Binop): 
    @classmethod
    def symbol(cls) -> str:
        return "/"

    @classmethod
    def execute(cls, op1: int, op2: int) -> int:
        if op2 == 0:
            raise ArithmeticError("Division by zero unsupported")
        if op1 % op2 != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        return op1 // op2

    @classmethod
    def solve_for_op1(cls, op2: int, result: int) -> int:
        if op2 == 0:
            raise ArithmeticError("Division by zero unsupported")
        return result * op2

    @classmethod
    def solve_for_op2(cls, op1: int, result: int) -> int:
        if op1 == result == 0:
            raise ArithmeticError("0/x=0 is True for all nonzero x. Unsolvable.")
        if result == 0:
            raise ArithmeticError(f"{op1}/x=0 False for all x")
        if op1 == 0:
            raise ArithmeticError(f"0/x={result} False for all x")
        # remaining case: both op1 and result nonzero
        if op1 % result != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        return op1 // result

    @classmethod
    def solve_for_op1_expanded(cls, op2: int, result: int) -> str:
        if op2 == 0:
            raise ArithmeticError("Division by zero unsupported")
        return f"{result}*{op2}"

    @classmethod
    def solve_for_op2_expanded(cls, op1: int, result: int) -> str:
        if op1 == result == 0:
            raise ArithmeticError("0/x=0 is True for all nonzero x. Unsolvable.")
        if result == 0:
            raise ArithmeticError(f"{op1}/x=0 False for all x")
        if op1 == 0:
            raise ArithmeticError(f"0/x={result} False for all x")
        # remaining case: both op1 and result nonzero
        if op1 % result != 0:
            raise ArithmeticError("Sample pairs should only contain integer solutions")
        return f"{op1}/{result}"

In [26]:
# check for errors
for binop_name, binop in Binop.registry.items():
    print(f"Checking Binop {binop_name}")
    assert not inspect.isabstract(binop), f"{binop_name} is still an abstract class"
    test_str = ''
    try:
        test_str = binop.generate_x_solution(x_first = False, op = 11, result=99)
        test_str += "\n" + binop.generate_x_solution(x_first = True, op = 11, result=99) + "\n"
    except ArithmeticError:
        test_str = binop.generate_x_solution(x_first = False, op = 77, result=11)
        test_str += "\n" + binop.generate_x_solution(x_first = True, op = 12, result=4) + "\n"
    print(test_str)

Checking Binop Add
11+x=99,x=? The answer is 88 because 99-11 equals to 88.
x+11=99,x=? The answer is 88 because 99-11 equals to 88.

Checking Binop Minus
11-x=99,x=? The answer is -88 because 11-99 equals to -88.
x-11=99,x=? The answer is 110 because 99+11 equals to 110.

Checking Binop Multiply
11*x=99,x=? The answer is 9 because 99/11 equals to 9.
x*11=99,x=? The answer is 9 because 99/11 equals to 9.

Checking Binop Divide
77/x=11,x=? The answer is 7 because 77/11 equals to 7.
x/12=4,x=? The answer is 48 because 4*12 equals to 48.



# Main Loop generate pairs

In [27]:
pairs = []

minimum_samples_per_binop = math.ceil(minimum_samples / len(Binop.registry))
''' 
every binop has 3 types of subquestions:
Direct, e.g. 2 + 3 = ?
Indirect left, e.g. x + 2 = 3
Indirect right, e.g. 2 + x = 3  
'''
minimum_samples_per_binop_subtype = math.ceil(minimum_samples_per_binop / 3)

print(f"Minimum number of samples: {minimum_samples}")
print(f"Minimum number of samples for each binop: {minimum_samples_per_binop}")
print(f"Minimum number of samples for each binop subtype: {minimum_samples_per_binop_subtype}")

def generate_pair_questions(
        incorrect_sampling_function : Callable[[int, int], str], 
        correct_sampling_function : Callable[[int, int], str],
        target_num_pairs : int = minimum_samples_per_binop_subtype, 
        ) -> list[dict]:
    pairs = []
    num_pairs = 0
    while num_pairs < target_num_pairs:
        for i in range(100):
            for j in range(100):
                try:
                    pairs += [{
                        "negative": incorrect_sampling_function(i,j),
                        "positive": correct_sampling_function(i,j),
                    }]

                except ArithmeticError:
                    pass
        # remove duplicates
        pairs = list({(p["negative"], p["positive"]): p for p in pairs}.values())

        # if new elements present, compare against new number of elements. If no new elements, stop trying to add new elemtns
        if num_pairs < len(pairs):
            num_pairs = len(pairs)
        else: 
            print(f"No new pairs found. \nCurrent number of elements = {len(pairs)} \nTarget number of elements = {target_num_pairs}\n")
            break
    return pairs

for binop_name, binop in Binop.registry.items():
    pairs += generate_pair_questions(
        incorrect_sampling_function = binop.generate_no_rearrange_solution_incorrect,
        correct_sampling_function = binop.generate_no_rearrange_solution
    )
    pairs += generate_pair_questions(
        incorrect_sampling_function = partial(binop.generate_x_solution_incorrect, True), 
        correct_sampling_function = partial(binop.generate_x_solution, True)
    )
    pairs += generate_pair_questions(
        incorrect_sampling_function = partial(binop.generate_x_solution_incorrect, False), 
        correct_sampling_function = partial(binop.generate_x_solution, False)
    )
    # to generate pairs regarding "Sorry, I do not know" -- not used since already have 100 samples
    # pairs += generate_pair_questions(
    #     incorrect_sampling_function = binop.generate_no_rearrange_solution_incorrect_str,
    #     correct_sampling_function = binop.generate_no_rearrange_solution
    # )
    # pairs += generate_pair_questions(
    #     incorrect_sampling_function = partial(binop.generate_x_solution_incorrect_str, True), 
    #     correct_sampling_function = partial(binop.generate_x_solution, True)
    # )
    # pairs += generate_pair_questions(
    #     incorrect_sampling_function = partial(binop.generate_x_solution_incorrect_str, False), 
    #     correct_sampling_function = partial(binop.generate_x_solution, False)
    # )

print(f"Total number of samples: {len(pairs)}")
print("Random 10 samples:")
random.shuffle(pairs)
for x in (pairs[:10]):
    print(x.items())

Minimum number of samples: 100000
Minimum number of samples for each binop: 25000
Minimum number of samples for each binop subtype: 8334
Total number of samples: 113652
Random 10 samples:
dict_items([('negative', '30-24=? The answer is 4 because 30-24 equals 4.'), ('positive', '30-24=? The answer is 6 because 30-24 equals 6.')])
dict_items([('negative', '18/x=3,x=? The answer is -2 because 18/3 equals to -2.'), ('positive', '18/x=3,x=? The answer is 6 because 18/3 equals to 6.')])
dict_items([('negative', '64/x=64,x=? The answer is 8 because 64/64 equals to 8.'), ('positive', '64/x=64,x=? The answer is 1 because 64/64 equals to 1.')])
dict_items([('negative', '0/14=? The answer is -27 because 0/14 equals -27.'), ('positive', '0/14=? The answer is 0 because 0/14 equals 0.')])
dict_items([('negative', 'x/64=13,x=? The answer is 813 because 13*64 equals to 813.'), ('positive', 'x/64=13,x=? The answer is 832 because 13*64 equals to 832.')])
dict_items([('negative', '82*56=? The answer is 4

In [28]:
with open ("./pos_neg_pairs.json", "w") as file:
    json.dump(pairs, file, indent=4)